# Kết quả Bài toán 1 – Global Trending (Chương 4, mục 4.1)

Notebook gọi lại `compute_trending` và `hit_rate_at_k` trong `src/02_task1_trending.py`, vẽ biểu đồ ngay trong notebook. **Không lưu ảnh, không ghi MongoDB.** Đặt notebook trong `notebooks/` cạnh `src/` và `data/`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 110
DATA = Path("..") / "data" / "master_ken_data.csv"

In [ ]:
import importlib.util
from matplotlib.patches import Patch
spec = importlib.util.spec_from_file_location("task1", Path("..") / "src" / "02_task1_trending.py")
task1 = importlib.util.module_from_spec(spec); spec.loader.exec_module(task1)

H, ALPHA = 14, 0.5
d1 = task1.load_data(DATA)
REF = task1.reference_date(d1)
rk = task1.compute_trending(d1, REF, h=H, alpha=ALPHA)
rk["rank_qty"] = rk["qty_raw"].rank(ascending=False, method="first").astype(int)
COL = {"DRINK": "#1F6F78", "FOOD": "#D98B2B"}
print("REFERENCE_DATE =", REF.date(), "| số món xếp hạng:", len(rk), "|", rk["item_type"].value_counts().to_dict())
display(rk.head(10)[["rank", "item_name", "item_type", "qty_raw", "Q_norm", "R_norm", "trending_score"]].round(3))

### 6.1 Top 10 món theo Trending Score (Hình 4.x)

In [ ]:
def hbar(ax, d, title):
    d = d.iloc[::-1]
    ax.barh(d["item_name"], d["trending_score"], color=[COL[t] for t in d["item_type"]], height=0.68)
    for y, v in enumerate(d["trending_score"]):
        ax.text(v + 0.01, y, f"{v:.3f}", va="center", fontsize=9)
    ax.set_xlim(0, 1.12); ax.set_xlabel("Trending Score"); ax.set_title(title, loc="left"); ax.grid(axis="x", alpha=.25)

fig, ax = plt.subplots(figsize=(8, 4.6))
hbar(ax, rk.head(10), f"Top 10 món theo Trending Score (h = {H}, α = {ALPHA})")
ax.legend(handles=[Patch(color=COL["DRINK"], label="Đồ uống (DRINK)"), Patch(color=COL["FOOD"], label="Bánh/Snack (FOOD)")],
          loc="lower right", frameon=False)
plt.tight_layout(); plt.show()

### 6.2 Top 5 đồ uống và Top 5 bánh/snack (theo `rank_in_type`)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 3.8))
hbar(axs[0], rk[rk["item_type"] == "DRINK"].head(5), "Top 5 đồ uống")
hbar(axs[1], rk[rk["item_type"] == "FOOD"].head(5), "Top 5 bánh và snack")
plt.tight_layout(); plt.show()

### 6.3 Số lượng chuẩn hóa, doanh thu chuẩn hóa và Trending Score của Top 10

In [ ]:
t = rk.head(10).iloc[::-1]; y = np.arange(len(t)); w = 0.36
fig, ax = plt.subplots(figsize=(8, 4.8))
ax.barh(y + w/2, t["Q_norm"], w, color="#1F6F78", label="Số lượng chuẩn hóa Q̂")
ax.barh(y - w/2, t["R_norm"], w, color="#9BB7BC", label="Doanh thu chuẩn hóa R̂")
ax.scatter(t["trending_score"], y, marker="D", color="#B5382F", zorder=3, s=28, label="Trending Score")
ax.set_yticks(y); ax.set_yticklabels(t["item_name"]); ax.set_xlim(0, 1.05); ax.grid(axis="x", alpha=.25)
ax.set_xlabel("Giá trị trong [0, 1]"); ax.legend(frameon=False, loc="lower right")
plt.tight_layout(); plt.show()

# Tỷ lệ Q / số lượng thô: đo mức tác động của Time Decay (gần như hằng số → Time Decay ít làm đổi thứ hạng)
rec = (rk["Q"] / rk["qty_raw"])
print(f"Q / số lượng thô: min {rec.min():.3f}, max {rec.max():.3f}, TB {rec.mean():.3f}, độ lệch chuẩn {rec.std():.3f}")

### 6.4 Thay đổi thứ hạng so với danh sách “bán chạy nhất”

In [ ]:
top = rk.head(12)
fig, ax = plt.subplots(figsize=(8, 4.6))
for _, r in top.iterrows():
    col = "#B5382F" if r["rank"] < r["rank_qty"] else ("#1F6F78" if r["rank"] > r["rank_qty"] else "#888")
    ax.plot([0, 1], [r["rank_qty"], r["rank"]], color=col, lw=1.6, marker="o")
    ax.text(-0.04, r["rank_qty"], r["item_name"], ha="right", va="center", fontsize=9)
    ax.text(1.04, r["rank"], r["item_name"], ha="left", va="center", fontsize=9)
ax.set_ylim(15, 0.3); ax.set_xlim(-0.55, 1.55); ax.set_yticks([]); ax.set_xticks([0, 1])
ax.set_xticklabels(["Hạng theo số lượng\n(bán chạy nhất)", "Hạng theo Trending Score"])
ax.spines[["left", "top", "right"]].set_visible(False)
ax.set_title("Thay đổi thứ hạng của 12 món đầu bảng (đỏ: tăng; xanh: giảm; xám: giữ)", loc="left", fontsize=11)
plt.tight_layout(); plt.show()
print("Tương quan hạng Spearman (số lượng vs Trending):", round(rk[["rank", "rank_qty"]].corr(method="spearman").iloc[0, 1], 3))

### 6.5 Độ nhạy tham số và Hit Rate@K (Bảng 4.x)

In [ ]:
rows = []
for h, a in [(7, .5), (14, .5), (28, .5), (14, 0.0), (14, 1.0)]:
    x = task1.compute_trending(d1, REF, h=h, alpha=a)
    rows.append({"h": h, "α": a, "Top 5": ", ".join(x.head(5)["item_name"])})
display(pd.DataFrame(rows))

hr = []
for h, a in [(7, .5), (14, .5), (28, .5), (14, 1.0)]:
    for k in (5, 10):
        r = task1.hit_rate_at_k(d1, "2026-10-13", "2026-10-27", k, h, a)
        hr.append({"h": h, "α": a, "K": k, "Hit Rate Trending": round(r["hit_rate_trending"], 3),
                   "Hit Rate bán chạy nhất": round(r["hit_rate_baseline"], 3), "Số đơn kiểm tra": r["n_orders_test"]})
display(pd.DataFrame(hr))